In [1]:

import pandas as pd

# 讀取老師提供的模擬保險理賠資料
df = pd.read_csv("../data/raw/claims_raw.csv")

# 查看資料筆數與欄位數
print("資料形狀：", df.shape)

# 顯示前 5 筆資料
display(df.head())


資料形狀： (1230, 13)


,claim_id,policy_id,age,gender,city,product,income,premium,claim_date,hospital_days,claim_amount,ocr_confidence,status
0,C100501,P33781,43,M,新北市,醫療險,342000.0,17500.0,2026/08/14,9,45800.0,83.8,已核付
1,C100992,P84067,32,M,台南市,儲蓄險,1254000.0,142300.0,2026/05/28,5,24100.0,71.7,已核付
2,C100880,P60530,37,F,台北市,重大疾病險,258000.0,9300.0,2026-04-15,3,21500.0,94.7,已核付
3,C100517,P68616,40,M,台中市,醫療險,540000.0,18000.0,2026-07-02,5,80400.0,99.0,已核付
4,C101022,P80193,46,F,桃園市,醫療險,602000.0,32800.0,2026-06-30,8,NaN,52.1,已核付


In [2]:

# 組員 D：四種險種的理賠金額比較

# 建立分析用副本，不修改原始資料
df_report = df.copy()

# 依照理賠編號去除重複資料
df_report = df_report.drop_duplicates(subset="claim_id")

# 將理賠金額轉成數字，去除 NT$ 和逗號
df_report["claim_amount"] = (
    df_report["claim_amount"]
    .astype("string")
    .str.replace("NT$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_report["claim_amount"] = pd.to_numeric(
    df_report["claim_amount"], errors="coerce"
)

# 按險種計算有效筆數、中位數、平均數
comparison = df_report.groupby("product")["claim_amount"].agg(
    ["count", "median", "mean"]
)

# 計算平均數與中位數的比值
comparison["ratio"] = comparison["mean"] / comparison["median"]

# 顯示比較表
display(comparison.round(2))


,count,median,mean,ratio
product,,,,
儲蓄險,243,29500.0,171362.14,5.81
意外險,287,21300.0,84971.08,3.99
醫療險,470,30500.0,85125.96,2.79
重大疾病險,156,29150.0,42643.59,1.46


In [3]:

# 組員 D：Pearson 與 Spearman 相關性分析

# 使用清理過重複值和理賠金額格式的資料
df_corr = df_report.copy()

# 指定要分析的數值欄位
cols = [
    "income",
    "premium",
    "hospital_days",
    "claim_amount",
    "ocr_confidence"
]

# 確保欄位都是數值
for col in cols:
    df_corr[col] = pd.to_numeric(
        df_corr[col], errors="coerce"
    )

# 計算 Pearson 相關係數
pearson = df_corr[cols].corr(method="pearson")

# 計算 Spearman 相關係數
spearman = df_corr[cols].corr(method="spearman")

# 整理老師要求的三組比較
pairs = [
    ("income", "premium"),
    ("hospital_days", "claim_amount"),
    ("ocr_confidence", "claim_amount")
]

results = []

for a, b in pairs:
    results.append({
        "欄位組合": f"{a} 與 {b}",
        "Pearson": pearson.loc[a, b],
        "Spearman": spearman.loc[a, b]
    })

corr_table = pd.DataFrame(results)

display(corr_table.round(2))


,欄位組合,Pearson,Spearman
0,income 與 premium,0.67,0.68
1,hospital_days 與 claim_amount,0.03,0.29
2,ocr_confidence 與 claim_amount,-0.00,0.00
